# Неделя 8. Решающее дерево своими руками
**Семинар · многоклассовая классификация · CART-подобное дерево на NumPy**

На занятии построим классификационное дерево из пороговых условий, а затем исследуем его поведение на данных с шумом. Сначала разберём **механику обучения и предсказания**, затем — **качество обобщения и влияние сложности дерева**.

**Маршрут занятия:** данные → Gini и энтропия → лучшее разбиение → рекурсивный рост → прогноз → границы решений → три класса → эксперименты с `scikit-learn`.

В учебной реализации используем простые циклы по признакам и порогам: так виден алгоритм поиска. В ходе занятия методы `MyDecisionTreeClassifier` разбираются по порядку. После каждого этапа можно выполнить соответствующую проверку.

## 0. Подготовка

Нужны `numpy`, `matplotlib`, `pandas`, `scikit-learn`. Все данные генерируются локально; внешние файлы и интернет не требуются. Фиксируем генераторы и разбиение, чтобы результаты можно было воспроизвести.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_circles, load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score

np.set_printoptions(precision=3, suppress=True)
pd.set_option('display.precision', 3)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

## 1. Набор A: 30 объектов для проверки алгоритма

Объект описывается двумя числовыми признаками $x_1$ и $x_2$. Метка $1$ стоит в верхнем правом прямоугольнике, остальные точки имеют метку $0$. Специально используем простое правило, чтобы можно было **понять найденный split**.

Это учебный набор для проверки кода, **не** доказательство качества на новых данных.

In [ ]:
def small_grid():
    x1, x2 = np.meshgrid(np.arange(1, 7), np.arange(1, 6))
    X = np.column_stack([x1.ravel(), x2.ravel()]).astype(float)
    y = ((X[:, 0] >= 4) & (X[:, 1] >= 3)).astype(int)
    return X, y

def plot_points(X, y, *, ax=None, title=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4.6))
    for label, marker in [(0, 'o'), (1, 's')]:
        subset = y == label
        ax.scatter(X[subset, 0], X[subset, 1], label=f'Класс {label}',
                   marker=marker, s=68, alpha=0.9, edgecolors='black', linewidths=0.45)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.grid(alpha=0.17)
    ax.legend(frameon=True, loc='lower right')
    if title: ax.set_title(title)
    return ax

X_a, y_a = small_grid()
print('X:', X_a.shape, '| классы:', np.bincount(y_a))
fig, ax = plt.subplots(figsize=(7, 4.6))
plot_points(X_a, y_a, ax=ax, title='Набор A: 30 точек')
ax.set(xlim=(0.5, 6.5), ylim=(0.5, 5.5), xticks=np.arange(1, 7), yticks=np.arange(1, 6))
plt.show()

**Вопрос перед кодом.** Если всё дерево пока состоит из одного листа, какой класс оно будет предсказывать? Сколько объектов ошибочно получат этот класс на train? Какие две границы позволяют выделить прямоугольник с меткой 1?

### Полезные операции NumPy

`np.unique(y, return_inverse=True)` возвращает **исходные метки** (`classes_`) и их **внутренние индексы** `0, …, K−1`. Так классификатор сможет работать с метками `10, 20, 30` и названиями классов, а не только с `0` и `1`.

`np.bincount(y_encoded, minlength=K)` подсчитывает объекты всех **K классов**, включая классы, отсутствующие в конкретном листе. `np.unique(X[:, j])` помогает найти возможные пороги; булевы маски распределяют объекты по дочерним узлам.

In [ ]:
labels = np.array(['кот', 'пёс', 'кот', 'птица', 'пёс'])
classes, encoded = np.unique(labels, return_inverse=True)
print('Классы:', classes)
print('Индексы меток:', encoded)
print('Количество объектов:', np.bincount(encoded, minlength=len(classes)))
demo_x = np.array([1., 2., 3., 4.])
print('Кандидаты на порог:', (demo_x[:-1] + demo_x[1:]) / 2)

## 2. Создаём классификационное дерево

На этом этапе мы сами реализуем дерево с осевыми пороговыми разбиениями. В начале класс будет пустым — в течение занятия его методы появятся последовательно **в одной и той же ячейке**. После каждого этапа можно заново выполнить эту ячейку и проверить работу нового метода.

Будем хранить дерево в словаре `self.tree`: ключ — номер узла. Для узла `i` левый дочерний узел имеет номер `2*i + 1`, правый — `2*i + 2`. Внутренний узел будет хранить признак и порог; лист — класс большинства и вектор вероятностей.

**Порядок реализации:**

1. Конструктор: настройки алгоритма и пустой словарь дерева.
2. `_gini`, `_entropy`, `_impurity`: как измеряем неоднородность классов.
3. `_split_impurity` и `_best_split`: поиск признака и порога с максимальным уменьшением неоднородности.
4. `_make_leaf`: как получить класс большинства и вероятности внутри листа.
5. `_fit_node` и `fit`: рекурсивное построение и условия остановки.
6. `_reach_leaf`, `predict_single`, `predict`, `predict_proba`: проход нового объекта по готовому дереву.

Внутри класса метки будут представлены индексами `0, ..., K−1`; соответствие исходным значениям хранится в `classes_`. Поэтому алгоритм подойдёт для любого числа классов, включая текстовые метки.

**Выбор правила остановки.** Мы допускаем разбиение с **нулевым выигрышем**, если оно создаёт две непустые допустимые группы. Это позволяет изучать зависимости, где полезность первого шага проявляется только после следующего, например XOR. Рост при этом остаётся конечным, так как каждая ветвь получает меньше объектов, а глубина и размеры групп контролируются.

In [ ]:
class MyDecisionTreeClassifier:
    pass

### Этап 1–2. Конструктор и неоднородность

Начните с `__init__`: запишите `max_depth`, `min_samples_split`, `min_samples_leaf`, `criterion` и пустой словарь `tree`. Затем добавьте `_gini(y)` и `_entropy(y)`, используя `np.bincount` для подсчёта классов. Метод `_impurity` будет выбирать формулу по параметру `criterion`.

После добавления этих методов **повторно выполните ячейку класса** и запустите проверку. Мы специально используем три равновероятных класса: корректная реализация не должна зависеть от бинарности метки.

In [ ]:
sample_tree = MyDecisionTreeClassifier(criterion='gini')
print('Gini (3 равных класса):', sample_tree._gini(np.array([0, 1, 2])))
print('Entropy (3 равных класса):', sample_tree._entropy(np.array([0, 1, 2])))
print('Gini (чистый узел):', sample_tree._gini(np.array([2, 2, 2])))
assert np.isclose(sample_tree._gini(np.array([0, 1, 2])), 2/3)
assert np.isclose(sample_tree._entropy(np.array([0, 1, 2])), np.log2(3))

**Итог проверки.** При трёх равных классах Gini равен $1-3(1/3)^2=2/3$, а энтропия — $\log_2 3$. Чистая группа имеет нулевую неоднородность. Эти формулы не требуют, чтобы классов было ровно два.

### Этап 3. Находим лучшее разбиение

Один кандидат состоит из признака `feature` и порога `threshold`. Для каждого признака достаточно проверить середины между соседними различными значениями. Разбиение оцениваем через взвешенную неоднородность дочерних групп; выигрыш — разность «до минус после».

Добавьте `_split_impurity` и `_best_split`, затем перезапустите ячейку класса. Посмотрим, какое условие алгоритм предложит **для корня**, когда все 30 объектов находятся в одном узле.

In [ ]:
feature, threshold, gain = sample_tree._best_split(X_a, y_a)
assert (feature, threshold) == (0, 3.5)
print(f'Лучший первый split: x_{feature + 1} <= {threshold:g}; выигрыш Gini = {gain:.3f}')

fig, ax = plt.subplots(figsize=(7, 4.6))
plot_points(X_a, y_a, ax=ax, title='Первое найденное разбиение')
ax.axvline(threshold, linestyle='--', linewidth=2, label=f'x1 = {threshold:g}')
ax.set(xlim=(0.5, 6.5), ylim=(0.5, 5.5))
ax.legend(loc='lower right')
plt.show()

**Итог поиска.** Для сетки из 30 объектов первый порог проходит между значениями $x_1=3$ и $x_1=4$, то есть $x_1\le3.5$. На графике он создаёт вертикальную границу. Этот результат относится только к **корню**: в дочерних группах дальше появятся свои лучшие разбиения.

### Этап 4–5. Лист и рекурсивное построение

Теперь одного порога недостаточно: после разбиения нужно создать дочерние узлы и применить к каждому ту же процедуру. Метод `_make_leaf` сохраняет число объектов, индекс класса большинства и вектор вероятностей длины `K`. Метод `_fit_node` решает, остановиться или выбрать split, затем рекурсивно строит левую и правую ветви. `fit` кодирует исходные метки и запускает рост из корня.

Допишите методы в ту же ячейку класса и запустите следующую проверку. В напечатанном словаре можно проследить, почему левый лист отделился сразу, а справа появилось ещё одно разбиение.

In [ ]:
toy_tree = MyDecisionTreeClassifier(max_depth=3, criterion='gini').fit(X_a, y_a)
for node_id, node in sorted(toy_tree.tree.items()):
    print(f'Узел {node_id}:', node)
print('Число листьев:', sum(node[0] == toy_tree.LEAF for node in toy_tree.tree.values()))

**Как читать словарь узлов.** Корень имеет номер 0. Его левый дочерний узел — 1, правый — 2; у правого узла следующие номера 5 и 6. Корень отделяет 15 нулей, правый узел делит оставшиеся 15 объектов на 6 нулей и 9 единиц. Число внутренних узлов — два, листьев — три.

### Этап 6. Предсказание и вероятности

Во время `predict` разбиения больше не ищутся: новый объект начинает в корне и следует по сохранённым условиям до листа. Добавьте `_reach_leaf`, `predict_single`, `predict` и `predict_proba`. Последний метод возвращает **вектор из K вероятностей**, в порядке `classes_`.

После нового запуска ячейки класса заново обучите `toy_tree` (перезапустите предыдущую проверку) и проверьте три объекта. Один должен пройти в верхний правый лист, остальные — в листья класса 0.

In [ ]:
example = np.array([[5.0, 4.0], [2.0, 4.0], [5.0, 1.0]])
print('Метки классов:', toy_tree.classes_)
print('Прогнозы:', toy_tree.predict(example))
print('Вероятности:\n', toy_tree.predict_proba(example))
assert np.array_equal(toy_tree.predict(example), [1, 0, 0])
assert toy_tree.predict_proba(example).shape == (3, 2)

**Итог проверки прогноза.** Для каждой новой точки выбран ровно один путь от корня к листу. `predict` возвращает исходную метку, а `predict_proba` — доли классов обучающих объектов соответствующего листа. На этих простых данных листья чистые, но это не обязательное свойство решающего дерева.

**Дополнительная проверка — XOR.** Первое разбиение для четырёх XOR-объектов не уменьшает Gini, но после второго уровня получаются чистые листья. Посмотрим, что даёт разрешение нулевого выигрыша. Этот пример соединяет практику с ограничениями жадного поиска из лекции.

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])
xor_tree = MyDecisionTreeClassifier(max_depth=2).fit(X_xor, y_xor)
print('Выигрыш в корне:', xor_tree.tree[0][-1])
print('Предсказания:', xor_tree.predict(X_xor))
print('Глубина до листьев: 2; листьев:', sum(n[0] == xor_tree.LEAF for n in xor_tree.tree.values()))
assert np.array_equal(xor_tree.predict(X_xor), y_xor)

**Что мы увидели.** Нулевой выигрыш корня не означает, что вся следующая структура бесполезна: два последующих разбиения позволяют классифицировать XOR. Если потребовать **строго положительный** выигрыш при каждом шаге, такое дерево остановится раньше. Это не гарантия, что разрешение нулевых улучшений всегда полезно, а пример зависимости результата от правил остановки.

## 3. Смотрим на обученное дерево

Словарь `self.tree` хранит номера узлов и локальные решения, но его не очень удобно читать как целую модель. Следующие готовые функции визуализации построят две связанные картинки: **структуру узлов** и **области решений на плоскости**. Сопоставьте каждый внутренний узел с соответствующей границей на графике.

In [ ]:
def plot_custom_tree(model, *, figsize=(11, 4.5)):
    """Небольшое дерево в виде схемы; узлы model.tree, без Graphviz."""
    coords, order = {}, [0]
    def locate(node_id, depth):
        node = model.tree[node_id]
        if node[0] == model.LEAF:
            x = order[0]
            order[0] += 1
        else:
            x = (locate(2 * node_id + 1, depth + 1)
                 + locate(2 * node_id + 2, depth + 1)) / 2
        coords[node_id] = (x, -depth)
        return x
    locate(0, 0)
    fig, ax = plt.subplots(figsize=figsize)
    for node_id, node in model.tree.items():
        x, y = coords[node_id]
        if node[0] != model.LEAF:
            for child_id in (2 * node_id + 1, 2 * node_id + 2):
                cx, cy = coords[child_id]
                ax.plot([x, cx], [y - 0.08, cy + 0.08], color='0.55', zorder=1)
            text = (f'x{node[1] + 1} <= {node[2]:.2f}\n'
                    f'n={node[3]}, gain={node[4]:.3f}')
        else:
            text = (f'Класс {model.classes_[node[1]]}\nn={node[3]}\n'
                    f'P={np.round(node[2], 2)}')
        ax.text(x, y, text, ha='center', va='center', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.55', facecolor='0.94',
                          edgecolor='0.55'), zorder=3)
    ax.set(xlim=(-0.8, max(2, order[0] - 0.2)),
           ylim=(min(yy for _, yy in coords.values()) - 0.65, 0.65))
    ax.axis('off')
    return fig, ax


def plot_boundary(model, X, y, *, ax=None, title=None, step=0.055):
    """Граница и объекты для любого числа классов при двух признаках."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4.5))
    classes = np.unique(y)
    x_lo, x_hi = X[:, 0].min() - 0.35, X[:, 0].max() + 0.35
    y_lo, y_hi = X[:, 1].min() - 0.35, X[:, 1].max() + 0.35
    xx, yy = np.meshgrid(np.arange(x_lo, x_hi, step),
                         np.arange(y_lo, y_hi, step))
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    predictions = model.predict(grid)
    codes = np.searchsorted(classes, predictions).reshape(xx.shape)
    ax.contourf(xx, yy, codes, levels=np.arange(len(classes)+1)-0.5,
                alpha=0.16, cmap='tab10')
    markers = ['o', 's', '^', 'D', 'v', 'P']
    for idx, cls in enumerate(classes):
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1], label=str(cls),
                   marker=markers[idx % len(markers)], s=22, alpha=0.8,
                   edgecolors='black', linewidths=0.3)
    ax.set(xlim=(x_lo, x_hi), ylim=(y_lo, y_hi), xlabel='$x_1$', ylabel='$x_2$')
    ax.legend(title='Класс', loc='best', fontsize=8)
    if title: ax.set_title(title)
    return ax


fig, ax = plot_custom_tree(toy_tree)
ax.set_title('Наше дерево на наборе A', pad=15)
plt.show()
fig, ax = plt.subplots(figsize=(7, 4.6))
plot_boundary(toy_tree, X_a, y_a, ax=ax, title='Области прогнозов на наборе A', step=0.075)
plt.show()


**Что получилось.** Корневой порог создаёт вертикальную границу. Следующее условие действует только внутри правой ветви, поэтому горизонтальная граница не проходит через всю плоскость. В этом учебном наборе три листа чистые: их вероятности равны 0 или 1. На реальных данных листья могут оставаться смешанными.

## 4. Три класса: пример Iris

Набор Iris содержит измерения цветков трёх видов ирисов. Возьмём длину и ширину лепестка; исходными метками станут названия видов. Обучим наше дерево и `DecisionTreeClassifier` с одинаковой максимальной глубиной.

**Перед запуском:** проверьте, что в `classes_` находятся три названия видов, а `predict_proba` имеет три столбца. Посмотрите, совпадают ли области решений моделей. Если отличаются, это не означает автоматически ошибку одной из реализаций: существует несколько вариантов выбора порогов и разрешения равных выигрышей.

In [ ]:
iris = load_iris()
X_iris = iris.data[:, [2, 3]]
y_iris = iris.target_names[iris.target]  # исходные метки — строки

custom_iris = MyDecisionTreeClassifier(max_depth=3, criterion='gini').fit(X_iris, y_iris)
sklearn_iris = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42).fit(X_iris, y_iris)
print('Порядок классов:', custom_iris.classes_)
print('Форма predict_proba:', custom_iris.predict_proba(X_iris[:4]).shape)
print('Три первых прогноза:', custom_iris.predict(X_iris[:3]))
assert custom_iris.predict_proba(X_iris[:4]).shape == (4, 3)
assert np.allclose(custom_iris.predict_proba(X_iris[:4]).sum(axis=1), 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)
plot_boundary(custom_iris, X_iris, y_iris, ax=axes[0], title='Своё дерево, Iris', step=0.035)
plot_boundary(sklearn_iris, X_iris, y_iris, ax=axes[1], title='scikit-learn, Iris', step=0.035)
plt.show()
fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(sklearn_iris, feature_names=['petal length', 'petal width'],
          class_names=[str(v) for v in sklearn_iris.classes_], filled=True,
          max_depth=3, fontsize=9, ax=ax)
ax.set_title('Библиотечное трёхклассовое дерево')
plt.show()

**Что показывает Iris.** В отличие от бинарной сетки, теперь модель работает с тремя классами и возвращает в `predict_proba` три столбца в порядке `classes_`. Библиотечное дерево визуализируется функцией `plot_tree`. Отличия в отдельных порогах допустимы, если выбор определяется деталями реализации или равными выигрышами.

## 5. Шумные окружности: исследуем сложность

Теперь перейдём от простой сетки к более сложной задаче: две группы объектов расположены примерно на концентрических окружностях, но из-за шума частично пересекаются. Одна крупная прямоугольная область уже не описывает правило.

Данные разделим на **train, validation, test**. Train нужен для обучения, validation — для выбора глубины и других настроек, test останется нетронутым до итоговой оценки. Сначала посмотрите на расположение точек; затем сравним модели разной сложности.

In [ ]:
X_b, y_b = make_circles(n_samples=400, factor=0.45, noise=0.25,
                        random_state=42)
X_dev, X_test, y_dev, y_test = train_test_split(
    X_b, y_b, test_size=0.20, stratify=y_b, random_state=21)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=21)

assert (len(X_train), len(X_valid), len(X_test)) == (240, 80, 80)
print('Train / Validation / Test:', len(X_train), len(X_valid), len(X_test))
fig, ax = plt.subplots(figsize=(7, 4.6))
plot_points(X_train, y_train, ax=ax, title='Набор B: обучающая часть')
ax.set_aspect('equal', adjustable='box')
plt.show()

**Что видно на данных.** Классы примерно соответствуют двум окружностям, однако из-за шума некоторые области пересекаются. Поэтому более глубокое дерево может точнее запомнить train, не обязательно улучшая качество на validation.

### Практика A. Как глубина влияет на качество

Обучите несколько деревьев с различными `max_depth`. Для каждого запишите **train accuracy**, **validation accuracy** и число листьев. Постройте график двух accuracy от глубины и сравните границы решений при трёх настройках.

**Вопрос перед экспериментом:** если разрешить дереву расти глубже, обязано ли улучшиться качество на validation? Как меняется число мелких областей и почему train accuracy может продолжать расти?

In [ ]:
depths = [1, 2, 3, 4, 5, 6, 8, None]
depth_rows = []
for depth in depths:
    model = MyDecisionTreeClassifier(max_depth=depth, criterion='gini').fit(X_train, y_train)
    depth_rows.append({
        'max_depth': 'без лимита' if depth is None else str(depth),
        'train_accuracy': accuracy_score(y_train, model.predict(X_train)),
        'valid_accuracy': accuracy_score(y_valid, model.predict(X_valid)),
        'n_leaves': sum(n[0] == model.LEAF for n in model.tree.values())
    })
depth_table = pd.DataFrame(depth_rows)
display(depth_table)

fig, ax = plt.subplots(figsize=(8, 4.3))
ax.plot(range(len(depths)), depth_table['train_accuracy'], marker='o', label='Train')
ax.plot(range(len(depths)), depth_table['valid_accuracy'], marker='s', label='Validation')
ax.set_xticks(range(len(depths)), depth_table['max_depth'])
ax.set(xlabel='Максимальная глубина', ylabel='Accuracy', ylim=(0.45, 1.03),
       title='Качество при увеличении глубины')
ax.grid(alpha=0.2)
ax.legend()
plt.show()

**Как читать результаты.** Рост допустимой глубины обычно позволяет уменьшить тренировочную ошибку и создать больше листьев. Validation accuracy может достичь максимума раньше, чем дерево перестанет улучшаться на train. Для выбора глубины используйте validation, а не test; форма кривой относится к выбранной синтетической выборке.

In [ ]:
# Сравнение форм границы при разных допустимых глубинах.
# Для этих графиков достаточно обучить три дерева с указанной глубиной.
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), constrained_layout=True)
for ax, depth in zip(axes, [2, 4, None]):
    m = MyDecisionTreeClassifier(max_depth=depth).fit(X_train, y_train)
    name = 'без лимита' if depth is None else f'depth={depth}'
    plot_boundary(m, X_train, y_train, ax=ax,
                  title=f'{name}; validation={accuracy_score(y_valid, m.predict(X_valid)):.3f}',
                  step=0.055)
plt.show()

**Что показывают границы.** На неглубокой модели видны крупные прямоугольные области; при большой глубине могут появиться узкие полосы и маленькие участки, подогнанные под отдельные обучающие точки. Внешний вид помогает понять механизм, но качество подтверждается только независимой выборкой.

### Практика B. Минимальное число объектов в листе

Параметр `min_samples_leaf` запрещает split, если хотя бы одна дочерняя группа слишком мала. Сравните значения 1, 5 и 20 при одинаковой `max_depth=8`. Измерьте число листьев, train/validation accuracy и посмотрите на границы.

**Вопрос перед экспериментом:** чем отличается «не делить маленький текущий узел» (`min_samples_split`) от «не создавать маленькие дочерние группы» (`min_samples_leaf`)?

In [ ]:
leaf_rows = []
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), constrained_layout=True)
for ax, leaf_size in zip(axes, [1, 5, 20]):
    model = MyDecisionTreeClassifier(max_depth=8, min_samples_leaf=leaf_size).fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    valid_acc = accuracy_score(y_valid, model.predict(X_valid))
    leaves = sum(n[0] == model.LEAF for n in model.tree.values())
    leaf_rows.append({'min_samples_leaf': leaf_size, 'leaves': leaves,
                      'train_accuracy': train_acc, 'valid_accuracy': valid_acc})
    plot_boundary(model, X_train, y_train, ax=ax,
                  title=f'min_samples_leaf={leaf_size} | {leaves} листьев', step=0.055)
plt.show()
display(pd.DataFrame(leaf_rows))

**Итог исследования размера листа.** Увеличение `min_samples_leaf` запрещает некоторые мелкие разбиения и обычно уменьшает число листьев. Это может улучшить устойчивость модели, но чрезмерное ограничение не позволит уловить настоящую зависимость — сравнивайте оба вида ошибки.

### Практика C. Gini и энтропия

Меняем только `criterion`, оставляя данные и ограничения сложности одинаковыми. Сравните первое условие дерева, число листьев, границы решений и validation accuracy при `max_depth=4`.

**Вопрос перед экспериментом:** если два критерия дают одинаковую accuracy, обязательно ли они нашли одно и то же дерево?

In [ ]:
criterion_rows = []
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.3), constrained_layout=True)
for ax, criterion in zip(axes, ['gini', 'entropy']):
    model = MyDecisionTreeClassifier(max_depth=4, criterion=criterion).fit(X_train, y_train)
    root = model.tree[0]
    criterion_rows.append({'criterion': criterion,
                           'root_condition': f'x{root[1]+1} <= {root[2]:.3f}',
                           'leaves': sum(n[0] == model.LEAF for n in model.tree.values()),
                           'valid_accuracy': accuracy_score(y_valid, model.predict(X_valid))})
    plot_boundary(model, X_train, y_train, ax=ax,
                  title=f'{criterion}: {criterion_rows[-1]["root_condition"]}', step=0.055)
plt.show()
display(pd.DataFrame(criterion_rows))

**Итог сравнения критериев.** Обе функции измеряют неоднородность, но по-разному реагируют на классовые доли. Поэтому найденные пороги и число листьев могут различаться. Одинаковая validation accuracy не доказывает тождественность деревьев.

## 6. Сопоставление с `DecisionTreeClassifier`

Наш класс реализует тот же общий принцип рекурсивного выбора разбиений, что и библиотечные деревья, но написан простыми циклами. Сравним обе реализации на одном наборе и одинаковой максимальной глубине. Посмотрим **validation accuracy**, текстовое представление условий через `export_text` и штатную схему `plot_tree`.

Обращайте внимание не только на accuracy: посмотрите, какие признаки используются в корне, какое число объектов указано в узлах и что показывает вектор `value` в библиотечной визуализации.

In [ ]:
sklearn_tree = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
sklearn_tree.fit(X_train, y_train)
our_tree = MyDecisionTreeClassifier(max_depth=3, criterion='gini').fit(X_train, y_train)

print('Validation accuracy:')
print('Наше дерево:', round(accuracy_score(y_valid, our_tree.predict(X_valid)), 3))
print('scikit-learn:', round(accuracy_score(y_valid, sklearn_tree.predict(X_valid)), 3))
print('\nПравила библиотечного дерева (до глубины 2):')
print(export_text(sklearn_tree, feature_names=['x1', 'x2'], max_depth=2))
fig, ax = plt.subplots(figsize=(13, 5))
plot_tree(sklearn_tree, feature_names=['x1', 'x2'], class_names=['0', '1'],
          filled=True, impurity=True, proportion=False, fontsize=9,
          max_depth=2, ax=ax)
ax.set_title('Библиотечное дерево (показаны первые уровни)')
plt.show()

**Что видно в `plot_tree`.** Внутренние узлы содержат условия по признакам, Gini и число обучающих объектов, а в листьях показаны значения целевой переменной, использованные для прогнозирования. Эта визуализация описывает **обученную структуру**, а не формулирует правила заранее.

**Как интерпретировать сравнение.** Одинаковая accuracy не означает одинакового дерева. В нашем коде перебираются середины соседних уникальных значений; при равном выигрыше сохраняется первый найденный вариант. В `scikit-learn` могут отличаться порядок рассмотрения признаков, точность порогов и правила разрешения равенств. При стандартном `min_impurity_decrease=0` библиотечное дерево, как и наше, может допускать split с нулевым выигрышем. Размеры листьев и глубина помогают сопоставить сложность моделей.

## 7. Как ошибки в метках влияют на дерево

Оставим признаки неизменными, но случайно заменим **18% обучающих меток** на противоположные. Обучим два библиотечных дерева с одинаковыми настройками и сравним их корневые условия, границы и качество на исходной validation-выборке с правильными метками.

**Предположение перед запуском:** дерево ориентируется на размеченные обучающие объекты; если метки меняются, лучшими могут оказаться другие пороги и даже другие корневые признаки. Но это не обязано происходить при любой небольшой ошибке.

In [ ]:
rng = np.random.default_rng(203)
y_corrupt = y_train.copy()
changed = rng.choice(len(y_corrupt), size=round(0.18 * len(y_corrupt)), replace=False)
y_corrupt[changed] = 1 - y_corrupt[changed]  # именно для бинарного набора B

clean = DecisionTreeClassifier(max_depth=5, random_state=42).fit(X_train, y_train)
noisy = DecisionTreeClassifier(max_depth=5, random_state=42).fit(X_train, y_corrupt)
print('Изменённых меток:', len(changed))
for name, model in [('Обычные метки', clean), ('18% изменённых меток', noisy)]:
    print(name, '| корень:', model.tree_.feature[0],
          '| train:', round(accuracy_score(y_train, model.predict(X_train)), 3),
          '| validation:', round(accuracy_score(y_valid, model.predict(X_valid)), 3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), constrained_layout=True)
plot_boundary(clean, X_train, y_train, ax=axes[0], title='Обучали по правильным меткам')
plot_boundary(noisy, X_train, y_train, ax=axes[1], title='18% меток на train изменены')
plt.show()


**Как интерпретировать изменение меток.** Изменённые метки влияют на подсчёт неоднородностей, поэтому при одинаковых признаках могут возникнуть другие разбиения. Снижение validation accuracy показывает возможную цену ошибок разметки, но не является гарантированным следствием любой порчи меток.

## 8. Чувствительность к составу обучающей выборки

Создадим несколько **bootstrap-выборок**: случайно выбираем обучающие объекты с возвращением, сохраняя исходное количество наблюдений. На каждой выборке обучим библиотечное дерево одинаковой глубины.

**Вопрос перед запуском:** совпадут ли верхние условия деревьев, обученных на разных вариантах данных? Почему близкое качество на validation может сочетаться с разными наборами правил?

In [ ]:
bootstrap_rows = []
fig, axes = plt.subplots(1, 3, figsize=(14, 4.0), constrained_layout=True)
for ax, seed in zip(axes, [4, 21, 77]):
    rng = np.random.default_rng(seed)
    ids = rng.choice(len(X_train), size=len(X_train), replace=True)
    model = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train[ids], y_train[ids])
    root = f'x{model.tree_.feature[0]+1} <= {model.tree_.threshold[0]:.2f}'
    val = accuracy_score(y_valid, model.predict(X_valid))
    bootstrap_rows.append({'seed': seed, 'корневой split': root,
                           'листьев': model.get_n_leaves(), 'validation accuracy': val})
    plot_boundary(model, X_train, y_train, ax=ax, title=f'seed={seed}; {root}')
plt.show()
display(pd.DataFrame(bootstrap_rows))


**Что показывает bootstrap.** Даже при одинаковом алгоритме и настройках разные обучающие выборки могут привести к различным порогам и геометрии решений. Устойчивость структуры и качество прогноза — не одно и то же; этот эффект важен для дальнейшего изучения ансамблей деревьев.

### Дополнительное исследование: post-pruning

В собственной реализации мы ограничивали рост дерева заранее. Библиотека также позволяет сначала построить большое дерево, а затем выбрать его упрощение. `cost_complexity_pruning_path` даёт последовательность значений `ccp_alpha`, соответствующую возможным шагам обрезки.

Сравните, как меняются количество листьев и качество на train/validation при увеличении штрафа за сложность. Почему минимальное дерево не обязательно должно иметь максимальную validation accuracy?

In [ ]:
path = DecisionTreeClassifier(random_state=42).cost_complexity_pruning_path(X_train, y_train)
positive = np.unique(path.ccp_alphas[path.ccp_alphas > 0])
alphas = [0.0, float(np.quantile(positive, 0.5)), float(np.quantile(positive, 0.85))]
rows = []
for a in alphas:
    model = DecisionTreeClassifier(random_state=42, ccp_alpha=a).fit(X_train, y_train)
    rows.append({'ccp_alpha': a, 'листьев': model.get_n_leaves(),
                 'train accuracy': accuracy_score(y_train, model.predict(X_train)),
                 'validation accuracy': accuracy_score(y_valid, model.predict(X_valid))})
display(pd.DataFrame(rows))

**Итог post-pruning.** С увеличением `ccp_alpha` выбор дерева обычно сдвигается в сторону меньшего числа листьев. Слишком большое значение штрафа может ухудшить качество и на train, и на validation; параметр выбирают по validation/CV, не по test.

## 9. Итоговая оценка на test

Выберите `max_depth` по validation accuracy из практики A. Если несколько вариантов показывают одинаковое качество, используйте заранее сформулированное правило, например выбор меньшей глубины. После этого обучите модель с выбранной глубиной на всех данных разработки (**train + validation**) и единственный раз оцените её на test.

Test не должен участвовать в выборе настроек. Иначе итоговая оценка перестанет быть независимой от процедуры подбора.

In [ ]:
# Глубину выбираем по validation accuracy;
# остальные настройки фиксированы: gini, min_samples_leaf=1.
best_idx = int(depth_table['valid_accuracy'].to_numpy().argmax())
best_depth_str = depth_table.loc[best_idx, 'max_depth']
best_depth = None if best_depth_str == 'без лимита' else int(best_depth_str)
print('Глубина, выбранная по validation:', best_depth)

final_model = MyDecisionTreeClassifier(max_depth=best_depth, criterion='gini')
final_model.fit(X_dev, y_dev)
final_test_accuracy = accuracy_score(y_test, final_model.predict(X_test))
print(f'Единственная оценка на test: accuracy = {final_test_accuracy:.3f}')

**Итог финальной оценки.** После выбора глубины по validation мы использовали все данные разработки для переобучения модели и ровно один раз оценили её на test. Эта цифра характеризует качество выбранной процедуры на данной отложенной выборке; подбирать настройки по ней уже нельзя.

## 10. Итоги

Мы реализовали дерево: считаем неоднородность, выбираем разбиение, рекурсивно строим структуру и записываем прогнозы в листья. Затем исследовали глубину, размер листьев, критерии и устойчивость дерева на данных.

**Вопросы для самопроверки:**

1. Зачем во взвешенной неоднородности нужны размеры дочерних групп?
2. Почему рекурсивное построение после первого split работает с подмножествами объектов?
3. Чем `min_samples_split` отличается от `min_samples_leaf`?
4. Почему train accuracy может расти, а validation accuracy снижаться?
5. Что меняется при переходе от Gini к entropy? Что остаётся прежним?
6. Что изменилось в реализации при переходе от двух классов к трём?
7. Почему разбиение с нулевым выигрышем иногда полезно? Рассмотрите XOR.
8. Почему test используют только после выбора настроек?

**Данные и воспроизводимость.** Набор A — синтетическая сетка из 30 объектов. Набор B генерируется через `make_circles` из `scikit-learn` (BSD-3-Clause). Генераторы и случайные разбиения используют фиксированные параметры `random_state`; реальные пользовательские данные не используются.